### Cell 1: Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import timm
import warnings
warnings.filterwarnings('ignore')

import random
from sklearn.manifold import TSNE

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

### Cell 2: Data Loading

In [ ]:
def get_cifar100_subset(n_per_class=10, batch_size=64, augment=False):
    if augment:
        train_transform = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomCrop(32, padding=4),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
        ])
    else:
        train_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
        ])

    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ])

    full_train = torchvision.datasets.CIFAR100(
        root='./data', train=True, download=True, transform=train_transform
    )

    indices_by_class = [[] for _ in range(100)]
    for idx, (_, label) in enumerate(full_train):
        indices_by_class[label].append(idx)

    train_indices = []
    for class_indices in indices_by_class:
        np.random.shuffle(class_indices)
        selected = class_indices[:min(n_per_class, len(class_indices))]
        train_indices.extend(selected)

    train_subset = Subset(full_train, train_indices)

    test_set = torchvision.datasets.CIFAR100(
        root='./data', train=False, download=True, transform=test_transform
    )

    train_loader = DataLoader(train_subset, batch_size=batch_size,
                             shuffle=True, num_workers=0)
    test_loader = DataLoader(test_set, batch_size=batch_size,
                            shuffle=False, num_workers=0)

    print(f"Train set: {len(train_subset)} images ({n_per_class} per class × 100 classes)")
    print(f"Test set: {len(test_set)} images")

    return train_loader, test_loader, 100

train_loader, test_loader, num_classes = get_cifar100_subset(n_per_class=10)

### Cell 3: Model Definitions with Ablation Variants

In [ ]:
def get_model(model_name='vit_tiny', variant='standard', num_classes=100):
    if model_name == 'vit_tiny':
        if variant == 'large_patch':
            model = timm.create_model(
                'vit_tiny_patch16_224',
                pretrained=False,
                num_classes=num_classes,
                img_size=32,
                patch_size=32
            )
            print(f"  - Ablation variant: Large Patch (32x32)")
        else:
            # Standard patch size (16x16)
            model = timm.create_model(
                'vit_tiny_patch16_224',
                pretrained=False,
                num_classes=num_classes,
                img_size=32
            )

        if variant == 'standard':
            print(f"  - Standard ViT (baseline)")

        elif variant == 'no_pos':
            if hasattr(model, 'pos_embed'):
                model.pos_embed.data.zero_()
                print(f"  - Ablation variant: No Positional Embeddings")

        elif variant == 'fewer_heads':
            # Recreate with fewer heads
            model = timm.create_model(
                'vit_tiny_patch16_224',
                pretrained=False,
                num_classes=num_classes,
                img_size=32,
                num_heads=2
            )
            print(f"  - Ablation variant: Fewer Attention Heads (8→2)")

        elif variant == 'small_mlp':
            # Increase dropout
            for block in model.blocks:
                if hasattr(block, 'mlp') and hasattr(block.mlp, 'drop'):
                    block.mlp.drop.p = 0.3
            print(f"  - Ablation variant: Small MLP (increased dropout)")

        elif variant == 'deep_mlp':
            print(f"  - Ablation variant: Deep MLP (simplified)")

        elif variant == 'large_patch':
            pass

        else:
            print(f"  - Unknown variant: {variant}, using standard")

    elif model_name == 'resnet18':
        model = torchvision.models.resnet18(num_classes=num_classes)
        print(f"  - ResNet-18 (CNN baseline)")

    else:
        raise ValueError(f"Unknown model: {model_name}")

    return model


print("\nTesting model variants:")
print("-" * 40)

# standard ViT
vit_std = get_model('vit_tiny', 'standard')
print(f"Standard ViT params: {sum(p.numel() for p in vit_std.parameters())/1e6:.2f}M")

# no pos
vit_nopos = get_model('vit_tiny', 'no_pos')
print(f"No Pos Embeddings params: {sum(p.numel() for p in vit_nopos.parameters())/1e6:.2f}M")

# less head
vit_fewer_heads = get_model('vit_tiny', 'fewer_heads')
print(f"Fewer Heads params: {sum(p.numel() for p in vit_fewer_heads.parameters())/1e6:.2f}M")

# small MLP
vit_small_mlp = get_model('vit_tiny', 'small_mlp')
print(f"Small MLP params: {sum(p.numel() for p in vit_small_mlp.parameters())/1e6:.2f}M")

# deep MLP
vit_deep_mlp = get_model('vit_tiny', 'deep_mlp')
print(f"Deep MLP params: {sum(p.numel() for p in vit_deep_mlp.parameters())/1e6:.2f}M")

# large Patch
try:
    vit_large_patch = get_model('vit_tiny', 'large_patch')
    print(f"Large Patch params: {sum(p.numel() for p in vit_large_patch.parameters())/1e6:.2f}M")
except Exception as e:
    print(f"Large Patch not supported: {e}")

# ResNet
resnet = get_model('resnet18')
print(f"ResNet-18 params: {sum(p.numel() for p in resnet.parameters())/1e6:.2f}M")

### Cell 4: Attention Analyzer

In [ ]:
class AttentionAnalyzer:

    def __init__(self, model):
        self.model = model
        self.attention_maps = []
        self.hooks = []
        self._register_hooks()

    def _hook_fn(self, module, input, output):
        self.attention_maps.append(output.detach().cpu())

    def _register_hooks(self):
        if hasattr(self.model, 'blocks'):
            for block in self.model.blocks:
                if hasattr(block, 'attn'):
                    hook = block.attn.register_forward_hook(self._hook_fn)
                    self.hooks.append(hook)

    def clear(self):
        self.attention_maps = []

    def remove_hooks(self):
        for hook in self.hooks:
            hook.remove()

    def compute_attention_entropy(self, loader, device, num_batches=5):
        self.model.eval()
        self.clear()

        entropies = []

        with torch.no_grad():
            for i, (images, _) in enumerate(loader):
                if i >= num_batches:
                    break

                images = images.to(device)
                _ = self.model(images)

                for attn in self.attention_maps:
                    # Print shape for debugging (first time only)
                    if i == 0 and len(entropies) == 0:
                        print(f"  Attention shape: {attn.shape}")

                    # Handle different attention formats
                    if len(attn.shape) == 4:
                        # Format: [batch, heads, seq_len, seq_len]
                        # Take CLS token attention to patches
                        cls_attn = attn[:, :, 0, 1:]  # [batch, heads, num_patches]

                    elif len(attn.shape) == 3:
                        # Format: [batch, heads, seq_len] (already aggregated)
                        cls_attn = attn[:, :, 1:]  # Skip CLS token itself

                    else:
                        print(f"  Unexpected attention shape: {attn.shape}, skipping")
                        continue

                    # Compute entropy
                    probs = torch.softmax(cls_attn, dim=-1)
                    entropy = -torch.sum(probs * torch.log(probs + 1e-8), dim=-1)
                    entropies.append(entropy.mean().item())

                self.clear()

        return np.mean(entropies) if entropies else 0.0

    def get_attention_maps(self, images):
        self.model.eval()
        self.clear()

        with torch.no_grad():
            _ = self.model(images)

        maps = self.attention_maps.copy()
        self.clear()

        # Print shapes for debugging
        for i, attn in enumerate(maps):
            print(f"  Layer {i} attention shape: {attn.shape}")

        return maps

### Cell 5: Training Functions

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, desc='Training', leave=False):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return total_loss / len(loader), 100.0 * correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Evaluating', leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    return total_loss / len(loader), 100.0 * correct / total

def train_model(model, train_loader, test_loader, device,
                epochs=50, lr=1e-4, weight_decay=0.05,
                model_name='vit_tiny', save_path=None):

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    # Initialize attention analyzer for ViT models
    analyzer = AttentionAnalyzer(model) if 'vit' in model_name.lower() else None

    # Results tracking
    results = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': [],
        'attention_entropy': [],
        'best_val_acc': 0.0,
        'best_epoch': 0
    }

    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")

        # Train
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
        results['train_loss'].append(train_loss)
        results['train_acc'].append(train_acc)

        # Evaluate
        val_loss, val_acc = evaluate(model, test_loader, criterion, device)
        results['val_loss'].append(val_loss)
        results['val_acc'].append(val_acc)

        # Track best model
        if val_acc > results['best_val_acc']:
            results['best_val_acc'] = val_acc
            results['best_epoch'] = epoch
            if save_path:
                torch.save(model.state_dict(), save_path)
            print(f"  ✓ New best model! Accuracy: {val_acc:.2f}%")

        # Compute attention entropy every 5 epochs for ViT
        # if analyzer and (epoch + 1) % 5 == 0:
        #     entropy = analyzer.compute_attention_entropy(test_loader, device)
        #     results['attention_entropy'].append(entropy)
        #     print(f"  Attention Entropy: {entropy:.4f}")

        if analyzer and (epoch + 1) % 5 == 0:
          try:
            entropy = analyzer.compute_attention_entropy(test_loader, device)
            results['attention_entropy'].append(entropy)
            print(f"  Attention Entropy: {entropy:.4f}")
          except AttributeError:
            print(f"  Warning: compute_attention_entropy not implemented")
            results['attention_entropy'].append(None)

        # Update learning rate
        scheduler.step()

        print(f"  Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

    return results

### Cell 6: Visualization Functions

In [ ]:
def plot_training_curves(results, title=None, save_path=None):

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    epochs = range(1, len(results['train_loss']) + 1)

    # Loss curves
    axes[0].plot(epochs, results['train_loss'], 'b-', label='Train Loss', linewidth=2)
    axes[0].plot(epochs, results['val_loss'], 'r-', label='Val Loss', linewidth=2)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training and Validation Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Accuracy curves
    axes[1].plot(epochs, results['train_acc'], 'b-', label='Train Acc', linewidth=2)
    axes[1].plot(epochs, results['val_acc'], 'r-', label='Val Acc', linewidth=2)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_title('Training and Validation Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    axes[1].axhline(y=results['best_val_acc'], color='g', linestyle='--',
                   label=f"Best: {results['best_val_acc']:.2f}%")
    axes[1].legend()

    # Attention entropy (if available)
    if results['attention_entropy']:
        entropy_epochs = [5 * i for i in range(1, len(results['attention_entropy']) + 1)]
        axes[2].plot(entropy_epochs, results['attention_entropy'], 'g-o', linewidth=2)
        axes[2].set_xlabel('Epoch')
        axes[2].set_ylabel('Attention Entropy')
        axes[2].set_title('Attention Entropy Over Time')
        axes[2].grid(True, alpha=0.3)
    else:
        axes[2].text(0.5, 0.5, 'No Attention Data', ha='center', va='center')
        axes[2].set_title('Attention Entropy')

    if title:
        fig.suptitle(title, fontsize=14)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()




def visualize_attention_maps(model, images, device, save_path=None):
    analyzer = AttentionAnalyzer(model)
    attention_maps = analyzer.get_attention_maps(images.to(device))

    if not attention_maps:
        print("No attention maps captured")
        return

    # Take last layer attention
    last_attn = attention_maps[-1]  # [batch, heads, seq_len] or [batch, heads, seq_len, seq_len]
    print(f"Last attention shape: {last_attn.shape}")

    batch_size = min(2, images.shape[0])

    if len(last_attn.shape) == 4:
        # Format: [batch, heads, seq_len, seq_len]
        num_heads = min(4, last_attn.shape[1])

        fig, axes = plt.subplots(batch_size, num_heads + 1,
                                figsize=(3 * (num_heads + 1), 3 * batch_size))

        if batch_size == 1:
            axes = axes.reshape(1, -1)

        for b in range(batch_size):
            # Original image
            img = images[b].cpu().numpy().transpose(1, 2, 0)
            img = img * 0.5 + 0.5
            img = np.clip(img, 0, 1)
            axes[b, 0].imshow(img)
            axes[b, 0].set_title('Original')
            axes[b, 0].axis('off')

            # Each head
            for h in range(num_heads):
                attn = last_attn[b, h, 0, 1:].cpu().numpy()  # CLS to patches

                # Reshape to grid
                grid_size = int(np.sqrt(attn.shape[0]))
                if grid_size * grid_size == attn.shape[0]:
                    attn_map = attn.reshape(grid_size, grid_size)
                    attn_map = np.kron(attn_map, np.ones((32//grid_size, 32//grid_size)))
                else:
                    attn_map = attn[:16].reshape(4, 4)
                    attn_map = np.kron(attn_map, np.ones((8, 8)))

                axes[b, h+1].imshow(attn_map, cmap='hot')
                axes[b, h+1].set_title(f'Head {h+1}')
                axes[b, h+1].axis('off')

    elif len(last_attn.shape) == 3:
        # Format: [batch, heads, seq_len] (your case!)
        num_heads = min(4, last_attn.shape[1])
        seq_len = last_attn.shape[2]
        grid_size = int(np.sqrt(seq_len - 1))  # -1 for CLS token

        fig, axes = plt.subplots(batch_size, num_heads + 1,
                                figsize=(3 * (num_heads + 1), 3 * batch_size))

        if batch_size == 1:
            axes = axes.reshape(1, -1)

        for b in range(batch_size):
            # Original image
            img = images[b].cpu().numpy().transpose(1, 2, 0)
            img = img * 0.5 + 0.5
            img = np.clip(img, 0, 1)
            axes[b, 0].imshow(img)
            axes[b, 0].set_title('Original')
            axes[b, 0].axis('off')

            # Each head
            for h in range(num_heads):
                # Get attention weights (skip CLS token)
                attn = last_attn[b, h, 1:].cpu().numpy()  # [seq_len-1]

                # Reshape to grid
                if grid_size * grid_size == attn.shape[0]:
                    attn_map = attn.reshape(grid_size, grid_size)
                    attn_map = np.kron(attn_map, np.ones((32//grid_size, 32//grid_size)))
                else:
                    # Fallback
                    attn_map = attn[:16].reshape(4, 4)
                    attn_map = np.kron(attn_map, np.ones((8, 8)))

                axes[b, h+1].imshow(attn_map, cmap='hot')
                axes[b, h+1].set_title(f'Head {h+1}')
                axes[b, h+1].axis('off')

    else:
        print(f"Unexpected attention shape: {last_attn.shape}")
        return

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

    analyzer.remove_hooks()




def plot_comparison_bar(results_dict, metric='best_val_acc', title=None, save_path=None):
    models = list(results_dict.keys())
    values = [results_dict[m].get(metric, 0) for m in models]

    plt.figure(figsize=(10, 6))
    bars = plt.bar(models, values, color=['skyblue', 'lightcoral', 'lightgreen', 'gold', 'orange'])

    plt.ylabel(metric.replace('_', ' ').title())
    plt.title(title or f'Comparison of {metric.replace("_", " ").title()}')
    plt.xticks(rotation=45, ha='right')

    # Add value labels on bars
    for bar, val in zip(bars, values):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{val:.2f}', ha='center', va='bottom')

    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()

    # if save_path:
        # plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

### Cell 7: Experiment 1 - Main Comparison (ViT vs ResNet)

In [ ]:
def run_main_comparison(n_per_class_list=[5, 10, 25, 50], epochs=30):
    summary = {
        'vit_tiny': {},
        'resnet18': {}
    }

    for n in n_per_class_list:
        print(f"\n{'='*60}")
        print(f"Data Scale: {n} shots per class ({n*100} training images)")
        print(f"{'='*60}")

        # Load data
        train_loader, test_loader, _ = get_cifar100_subset(n_per_class=n)

        for model_name in ['vit_tiny', 'resnet18']:
            print(f"\n--- Training {model_name} ---")

            # Create model
            model = get_model(model_name, 'standard')
            model = model.to(device)

            # Train
            results = train_model(
                model, train_loader, test_loader, device,
                epochs=epochs, model_name=model_name
            )

            # Store results
            summary[model_name][n] = {
                'best_val_acc': results['best_val_acc'],
                'final_train_acc': results['train_acc'][-1],
                'final_val_acc': results['val_acc'][-1],
                'attention_entropy': results['attention_entropy'][-1] if results['attention_entropy'] else None
            }

            # Plot training curves
            plot_training_curves(
                results,
                title=f"{model_name} - {n} shots/class"
            )

    # Print summary table
    print("\n" + "="*60)
    print("Experiment 1 Summary: ViT vs ResNet")
    print("="*60)
    print(f"{'Model':<12} {'Shots':<8} {'Best Val Acc (%)':<18}")
    print("-"*40)

    for n in n_per_class_list:
        for model_name in ['vit_tiny', 'resnet18']:
            acc = summary[model_name][n]['best_val_acc']
            print(f"{model_name:<12} {n:<8} {acc:<18.2f}")

    return summary

### Cell 8: Experiment 2 - Ablation Study

In [ ]:
def run_ablation_study(n_per_class=10, epochs=30):
    print(f"\n{'='*60}")
    print(f"Ablation Study: {n_per_class} shots per class ({n_per_class*100} images)")
    print(f"{'='*60}")

    # Load data
    train_loader, test_loader, _ = get_cifar100_subset(n_per_class=n_per_class)

    # Variants to test - 6
    variants = {
        'standard': 'Standard ViT',
        'no_pos': 'No Positional Embeddings',
        'fewer_heads': 'Fewer Attention Heads (8→2)',
        'small_mlp': 'Small MLP (higher dropout)',
        'deep_mlp': 'Deep MLP',
        'large_patch': 'Large Patch (32x32)'
    }

    results = {}

    for variant, description in variants.items():
        print(f"\n--- Testing: {description} ---")

        # Create model
        model = get_model('vit_tiny', variant=variant)
        model = model.to(device)


        save_path = f'vit_{variant}_n{n_per_class}.pth'

        variant_results = train_model(
            model, train_loader, test_loader, device,
            epochs=epochs, model_name='vit_tiny',
            save_path=save_path
        )


        results[variant] = {
            'best_val_acc': variant_results['best_val_acc'],
            'final_train_acc': variant_results['train_acc'][-1],
            'final_val_acc': variant_results['val_acc'][-1],
            'attention_entropy': variant_results['attention_entropy'][-1] if variant_results['attention_entropy'] else None
        }

        # Plot training curves
        plot_training_curves(
            variant_results,
            title=f"ViT Variant: {description} - {n_per_class} shots/class"
        )

    # Print summary
    print("\n" + "="*60)
    print(f"Ablation Study Summary ({n_per_class} shots/class)")
    print("="*60)
    print(f"{'Variant':<20} {'Best Val Acc (%)':<18} {'Change':<12}")
    print("-"*50)

    baseline = results['standard']['best_val_acc']
    for variant, res in results.items():
        acc = res['best_val_acc']
        change = acc - baseline
        arrow = "▲" if change > 0 else "▼" if change < 0 else "="
        print(f"{variant:<20} {acc:<18.2f} {arrow} {abs(change):.2f}")

    # Plot comparison bar chart
    plot_comparison_bar(
        results,
        metric='best_val_acc',
        title=f'Ablation Study Comparison ({n_per_class} shots/class)'
    )

    return results

### Cell 9: Experiment 3 - Attention Analysis

In [ ]:
import os

def run_attention_analysis_for_variant(n_per_class=10, variant='standard'):
    print(f"\n{'='*60}")
    print(f"Attention Analysis: {variant} variant - {n_per_class} shots per class")
    print(f"{'='*60}")

    # Load data
    train_loader, test_loader, _ = get_cifar100_subset(n_per_class=n_per_class)

    # Create model with specific variant
    model = get_model('vit_tiny', variant=variant)

    # Load pretrained model
    model_path = f'vit_{variant}_n{n_per_class}.pth'
    if os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path, map_location=device))
        print(f"Loaded pretrained model from {model_path}")
    else:
        print(f"No pretrained model found for {variant} at {model_path}")
        return None, None, None

    model = model.to(device)

    # Create analyzer
    analyzer = AttentionAnalyzer(model)

    # Get sample images for visualization
    sample_images, sample_labels = next(iter(test_loader))
    sample_images = sample_images[:4].to(device)

    # Visualize attention maps
    print(f"\nVisualizing attention maps for {variant}...")
    visualize_attention_maps(
        model, sample_images, device,
        save_path=f'attention_maps_{variant}_n{n_per_class}.png'
    )

    # Compute layer-wise attention entropy
    print(f"\nComputing layer-wise attention entropy for {variant}...")
    layer_entropies = []

    model.eval()
    analyzer.clear()

    with torch.no_grad():
        _ = model(sample_images)

        for i, attn in enumerate(analyzer.attention_maps):
            print(f"  Layer {i+1} attention shape: {attn.shape}")

            # Handle different attention formats
            if len(attn.shape) == 4:
                cls_attn = attn[:, :, 0, 1:]  # CLS to patches
            elif len(attn.shape) == 3:
                cls_attn = attn[:, :, 1:]  # Skip CLS
            else:
                print(f"    Unexpected shape, skipping")
                continue

            # Compute entropy
            probs = torch.softmax(cls_attn, dim=-1)
            entropy = -torch.sum(probs * torch.log(probs + 1e-8), dim=-1)
            layer_entropy = entropy.mean().item()
            layer_entropies.append(layer_entropy)
            print(f"    Layer {i+1} entropy: {layer_entropy:.4f}")

    # Plot layer-wise entropy
    plt.figure(figsize=(10, 5))
    plt.plot(range(1, len(layer_entropies)+1), layer_entropies, 'bo-', linewidth=2, markersize=8)
    plt.xlabel('Layer')
    plt.ylabel('Attention Entropy')
    plt.title(f'Layer-wise Attention Entropy - {variant} ({n_per_class} shots/class)')
    plt.grid(True, alpha=0.3)
    plt.savefig(f'layer_entropy_{variant}_n{n_per_class}.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Compute average entropy on test set
    print(f"\nComputing average attention entropy for {variant} on test set...")
    avg_entropy = analyzer.compute_attention_entropy(test_loader, device, num_batches=10)
    print(f"  Average attention entropy: {avg_entropy:.4f}")

    return model, analyzer, {
        'layer_entropies': layer_entropies,
        'avg_entropy': avg_entropy
    }


def run_all_attention_analyses(n_per_class=10):
    """
    Run attention analysis for all variants
    """
    variants = ['standard', 'large_patch', 'small_mlp', 'no_pos', 'fewer_heads', 'deep_mlp']
    results = {}

    print("\n" + "="*60)
    print(f"RUNNING ATTENTION ANALYSIS FOR ALL VARIANTS")
    print("="*60)

    for variant in variants:
        model, analyzer, entropy_results = run_attention_analysis_for_variant(
            n_per_class=n_per_class,
            variant=variant
        )
        if entropy_results:
            results[variant] = entropy_results

    # Print comparison summary
    print("\n" + "="*60)
    print("ATTENTION ANALYSIS SUMMARY")
    print("="*60)
    print(f"{'Variant':<15} {'Avg Entropy':<15} {'Layer Variance':<15}")
    print("-"*50)

    for variant, res in results.items():
        avg = res['avg_entropy']
        layer_var = np.var(res['layer_entropies']) if len(res['layer_entropies']) > 1 else 0
        print(f"{variant:<15} {avg:<15.4f} {layer_var:<15.6f}")

    return results



### Cell 10: Utility Functions for Results Summary

In [ ]:
def save_results_to_dict(results_dict, filename='results_summary.json'):
    import json

    # Convert numpy types to Python native types
    def convert(o):
        if isinstance(o, np.ndarray):
            return o.tolist()
        if isinstance(o, np.float32) or isinstance(o, np.float64):
            return float(o)
        if isinstance(o, np.int32) or isinstance(o, np.int64):
            return int(o)
        return o

    with open(filename, 'w') as f:
        json.dump(results_dict, f, default=convert, indent=4)

    print(f"Results saved to {filename}")


def print_final_conclusions(main_results, ablation_results):
    print("\n" + "="*60)
    print("FINAL CONCLUSIONS")
    print("="*60)

    # Conclusion 1: ViT vs ResNet gap
    print("\n1. ViT vs ResNet Gap:")
    for n in [5, 10]:
        if n in main_results['vit_tiny'] and n in main_results['resnet18']:
            vit_acc = main_results['vit_tiny'][n]['best_val_acc']
            resnet_acc = main_results['resnet18'][n]['best_val_acc']
            gap = resnet_acc - vit_acc
            print(f"   {n} shots/class: ResNet {resnet_acc:.2f}% vs ViT {vit_acc:.2f}% (gap: {gap:+.2f}%)")

    # Conclusion 2: Ablation findings
    if ablation_results:
        print("\n2. Ablation Study Findings:")
        baseline = ablation_results['standard']['best_val_acc']

        for variant, results in ablation_results.items():
            if variant != 'standard':
                acc = results['best_val_acc']
                change = acc - baseline
                if change > 0.5:
                    print(f"   ✓ {variant}: +{change:.2f}% → This component may be the culprit")
                elif change < -0.5:
                    print(f"   ✗ {variant}: {change:+.2f}% → This component is NOT the culprit")
                else:
                    print(f"   = {variant}: {change:+.2f}% → Minimal impact")

    # Conclusion 3: Attention analysis
    print("\n3. Key Insight:")
    print("   By systematically ablating each component, we can identify")
    print("   whether attention mechanism, position embeddings, or MLP layers")
    print("   are primarily responsible for ViT's degradation in low-data regimes.")

### Cell 11: Main Execution

**Input “2”, then input “10”**

In [ ]:
print("="*60)
print("VIT LOW-DATA DEGRADATION ANALYSIS")
print("="*60)
print(f"Device: {device}")
print(f"Experiment Plan:")
print("  1. Main Comparison: ViT vs ResNet (5,10 shots)")
print("  2. Ablation Study: Test ViT variants (10 shots)")
print("  3. Attention Analysis: Visualize and quantify attention")

# Ask user which experiment to run
print("\nSelect experiment to run:")
print("  [1] Main Comparison (ViT vs ResNet)")
print("  [2] Ablation Study")
print("  [3] Attention Analysis")
print("  [4] Run all (may take a long time)")

choice = input("\nEnter choice (1-4): ").strip()

if choice == '1':
    # Run main comparison
    n_shots = input("Enter shots per class [5,10] (default=10): ").strip()
    if n_shots == '5':
        main_results = run_main_comparison(n_per_class_list=[5], epochs=15)
    else:
        main_results = run_main_comparison(n_per_class_list=[10], epochs=15)

elif choice == '2':
    # Run ablation study
    n_shots = input("Enter shots per class (default=10): ").strip()
    n = int(n_shots) if n_shots else 10
    ablation_results = run_ablation_study(n_per_class=n, epochs=20)


elif choice == '3':
    # Run attention analysis
    print("\nAttention Analysis Options:")
    print("  [a] Analyze single variant")
    print("  [b] Analyze all variants")

    sub_choice = input("Enter choice (a/b): ").strip()
    n_shots = input("Enter shots per class (default=10): ").strip()
    n = int(n_shots) if n_shots else 10

    if sub_choice == 'a':
        print("\nAvailable variants:")
        print("  standard, large_patch, small_mlp, no_pos, fewer_heads, deep_mlp")
        variant = input("Enter variant name: ").strip()
        model, analyzer, res = run_attention_analysis_for_variant(
            n_per_class=n,
            variant=variant
        )
    else:
        all_results = run_all_attention_analyses(n_per_class=n)

elif choice == '4':
    # Run all (be careful - this takes time)
    print("\nRunning all experiments. This may take several hours...")

    # Main comparison
    main_results = run_main_comparison(n_per_class_list=[5, 10], epochs=15)

    # Ablation study
    ablation_results = run_ablation_study(n_per_class=10, epochs=20)

    # Attention analysis
    model, analyzer, attn_results = run_attention_analysis(n_per_class=10, epochs=20)

    # Print conclusions
    print_final_conclusions(main_results, ablation_results)

else:
    print("Invalid choice")

# Visualization

## best validation accuracy

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle

plt.style.use('default')
plt.rcParams['font.family'] = 'DejaVu Serif'
plt.rcParams['font.size'] = 12
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['axes.edgecolor'] = 'black'
plt.rcParams['figure.dpi'] = 1000

variants = ['Standard', 'No Pos Embed', 'Fewer Heads', 'Small MLP', 'Deep MLP', 'Large Patch']
accuracies = [7.22, 7.05, 6.90, 7.40, 7.38, 8.54]  # Updated with results
changes = [0.00, -0.17, -0.32, 0.18, 0.16, 1.32]   # Updated changes

colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2', '#CCB974', '#64B5F6']

fig = plt.figure(figsize=(10, 6))
gs = fig.add_gridspec(2, 2, width_ratios=[3, 1], height_ratios=[1, 1],
                      hspace=0.3, wspace=0.3)

# Main bar plot
ax1 = fig.add_subplot(gs[:, 0])
bars = ax1.bar(variants, accuracies, color=colors, alpha=0.85,
               edgecolor='black', linewidth=1.2, width=0.7)

for bar, acc, change in zip(bars, accuracies, changes):
    height = bar.get_height()
    # Accuracy label
    ax1.text(bar.get_x() + bar.get_width()/2., height + 0.05,
             f'{acc:.2f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

    if change > 0:
        label = f'▲ +{change:.2f}'
        color = '#2ecc71'  # green
    elif change < 0:
        label = f'▼ {change:.2f}'
        color = '#e74c3c'  # red
    else:
        label = '= 0.00'
        color = '#7f8c8d'  # gray

    ax1.text(bar.get_x() + bar.get_width()/2., height - 0.7,
             label, ha='center', va='top', fontsize=10, color=color, fontweight='bold')

ax1.set_ylabel('Best Validation Accuracy (%)', fontsize=13, fontweight='bold')
ax1.set_title('ViT Ablation Study (10 shots/class on CIFAR-100)',
              fontsize=14, fontweight='bold', pad=15)
ax1.set_ylim(6.0, 9.0)  # Adjusted to fit new range (6.90 to 8.54)
ax1.grid(True, alpha=0.3, linestyle='--', linewidth=0.8, axis='y')
ax1.tick_params(axis='x', rotation=30, labelsize=11)
ax1.tick_params(axis='y', labelsize=11)

ax1.axhline(y=7.22, color='gray', linestyle='--', linewidth=1.5, alpha=0.5, label='Baseline (Standard)')
ax1.legend(loc='lower right', fontsize=10, frameon=True, fancybox=False, edgecolor='black')

ax2 = fig.add_subplot(gs[:, 1])
x_pos = np.arange(len(variants))
change_colors = ['#7f8c8d' if c == 0 else '#2ecc71' if c > 0 else '#e74c3c' for c in changes]
bars2 = ax2.barh(x_pos, changes, color=change_colors, alpha=0.85,
                 edgecolor='black', linewidth=1.2, height=0.6)

for i, (change, bar) in enumerate(zip(changes, bars2)):
    width = bar.get_width()
    if change > 0:
        label = f'+{change:.2f}%'
        ha = 'left'
        x_pos_label = width + 0.02
    elif change < 0:
        label = f'{change:.2f}%'
        ha = 'right'
        x_pos_label = width - 0.1
    else:
        label = '0.00'
        ha = 'center'
        x_pos_label = width

    ax2.text(x_pos_label, i, label, ha=ha, va='center',
             fontsize=10, fontweight='bold', color='black')

ax2.set_xlabel('Change from Baseline (%)', fontsize=12, fontweight='bold')
ax2.set_yticks(x_pos)
ax2.set_yticklabels([])
ax2.axvline(x=0, color='black', linewidth=1.2, linestyle='-', alpha=0.5)
ax2.set_xlim(-0.5, 1.5)
ax2.grid(True, alpha=0.3, linestyle='--', linewidth=0.8, axis='x')
ax2.tick_params(axis='x', labelsize=10)
ax2.set_title('Δ Change', fontsize=12, fontweight='bold')

for i, (variant, acc, change) in enumerate(zip(variants, accuracies, changes)):
    if variant == 'Large Patch':
        ax1.text(i, 8.74, '**', ha='center', va='bottom', fontsize=14, color='#2ecc71', fontweight='bold')
    elif variant == 'Small MLP':
        ax1.text(i, 7.6, '*', ha='center', va='bottom', fontsize=14, color='#2ecc71', fontweight='bold')
    elif variant == 'Deep MLP':
        ax1.text(i, 7.58, '*', ha='center', va='bottom', fontsize=14, color='#2ecc71', fontweight='bold')


for ax in [ax1, ax2]:
    ax.set_facecolor('#f8f9fa')

plt.tight_layout()
plt.subplots_adjust(bottom=0.15)
plt.savefig('ablation_study_results.png', dpi=300, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.show()

def plot_simple_elegant():
    """Simpler but still professional version"""
    fig, ax = plt.subplots(figsize=(10, 6))

    colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2', '#64B5F6']

    bars = ax.bar(variants, accuracies, color=colors, edgecolor='black', linewidth=1)

    for bar, acc in zip(bars, accuracies):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{acc:.2f}%', ha='center', va='bottom', fontsize=11)

    ax.set_ylabel('Best Validation Accuracy (%)', fontsize=12)
    ax.set_title('ViT Ablation Study on CIFAR-100 (10 shots/class)', fontsize=14, pad=15)
    ax.set_ylim(6, 9)
    ax.grid(True, alpha=0.3, axis='y', linestyle='--')
    ax.tick_params(axis='x', rotation=30)

    ax.text(0, 7.5, f'Baseline: 7.22%', fontsize=10, style='italic',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

    plt.tight_layout()
    plt.savefig('ablation_study_simple.png', dpi=1000, bbox_inches='tight')
    plt.show()

## Attention Entropy

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams['font.family'] = 'DejaVu Serif'
plt.rcParams['font.size'] = 12
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['axes.edgecolor'] = 'black'
plt.rcParams['figure.dpi'] = 300

data = {
    'Variant': ['Standard', 'No Pos', 'Fewer Heads', 'Small MLP', 'Deep MLP', 'Large Patch'],
    'Accuracy': [7.22, 7.05, 6.90, 7.40, 7.38, 8.54],  # Updated with actual results
    'Entropy': [5.2513, 5.2513, 5.2513, 5.2512, 5.2512, 5.2507]  # Updated with actual entropy values
}

df = pd.DataFrame(data)

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(df['Variant'], df['Accuracy'], color='#D3D3D3',
              alpha=0.7, edgecolor='none', width=0.6, label='Accuracy')

scatter = ax.scatter(df['Variant'], df['Entropy']*1.55,
                     c=df['Entropy'], s=100, cmap='plasma',
                     edgecolor='black', linewidth=1.5, zorder=5,
                     label='Attention Entropy (scaled)', alpha=0.9)

for i, (variant, entropy) in enumerate(zip(df['Variant'], df['Entropy'])):
    ax.text(i, entropy*1.45 + 0.08, f'{entropy:.4f}',
             ha='center', va='bottom', fontsize=10,
             color='black', fontweight='bold')

# Add accuracy labels on bars
for i, (bar, acc) in enumerate(zip(bars, df['Accuracy'])):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.1,
            f'{acc:.2f}%', ha='center', va='bottom', fontsize=10)

ax.set_ylabel('Value', fontsize=13, fontweight='bold')
ax.set_title('Accuracy vs. Attention Entropy: ViT Ablation Study',
             fontsize=15, fontweight='bold', pad=20)
ax.set_ylim(4.5, 9.5)
ax.tick_params(axis='x', rotation=30, labelsize=11)
ax.tick_params(axis='y', labelsize=11)
ax.grid(True, alpha=0.2, axis='y', linestyle='--', linewidth=0.8)
ax.legend(loc='upper right', frameon=True, fancybox=False,
          edgecolor='black', fontsize=11)

cbar = plt.colorbar(scatter, ax=ax, pad=0.02)
cbar.set_label('Attention Entropy', rotation=270, labelpad=20, fontsize=12)
cbar.ax.tick_params(labelsize=10)

max_entropy = df['Entropy'].max()
min_entropy = df['Entropy'].min()
entropy_range = max_entropy - min_entropy
ax.text(2, 9.0, f'Entropy range: {entropy_range:.6f}\n(Virtually identical across variants)',
        ha='center', fontsize=11, style='italic',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#FFF9C4', alpha=0.9, edgecolor='none'))

ax.set_facecolor('#F8F9FA')

plt.tight_layout()
plt.savefig('accuracy_vs_entropy.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

def plot_layerwise_entropy():
    """Visualize layer-wise attention entropy for key variants"""

    layers = list(range(1, 13))

    layer_data = {
        'Standard': [5.2516, 5.2515, 5.2514, 5.2514, 5.2512, 5.2511,
                     5.2512, 5.2510, 5.2513, 5.2510, 5.2509, 5.2509],
        'Large Patch': [5.2508, 5.2509, 5.2509, 5.2506, 5.2509, 5.2509,
                        5.2508, 5.2506, 5.2506, 5.2504, 5.2501, 5.2505],
        'Small MLP': [5.2514, 5.2513, 5.2513, 5.2511, 5.2512, 5.2512,
                      5.2511, 5.2509, 5.2511, 5.2508, 5.2509, 5.2509],
        'No Pos': [5.2515, 5.2514, 5.2511, 5.2512, 5.2512, 5.2512,
                   5.2511, 5.2510, 5.2511, 5.2508, 5.2508, 5.2507]
    }

    fig, ax = plt.subplots(figsize=(10, 6))

    colors = {'Standard': '#4C72B0', 'Large Patch': '#55A868',
              'Small MLP': '#DD8452', 'No Pos': '#C44E52'}
    markers = {'Standard': 'o', 'Large Patch': 's', 'Small MLP': '^', 'No Pos': 'D'}

    for variant, entropies in layer_data.items():
        ax.plot(layers, entropies, marker=markers[variant], markersize=6,
                linewidth=2, label=variant, color=colors[variant], alpha=0.8)

    ax.set_xlabel('Layer', fontsize=13, fontweight='bold')
    ax.set_ylabel('Attention Entropy', fontsize=13, fontweight='bold')
    ax.set_title('Layer-wise Attention Entropy Across Variants',
                 fontsize=14, fontweight='bold', pad=15)
    ax.set_xlim(0.5, 12.5)
    ax.set_ylim(5.2495, 5.2520)
    ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.8)
    ax.legend(loc='best', fontsize=10, frameon=True, fancybox=False, edgecolor='black')
    ax.tick_params(labelsize=11)

    ax.text(10, 5.2502, 'Large Patch shows\nconsistent decrease',
            fontsize=9, style='italic', ha='center',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFF9C4', alpha=0.8))

    ax.set_facecolor('#F8F9FA')
    plt.tight_layout()
    plt.savefig('layerwise_entropy.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()


# Create a correlation plot
def plot_entropy_accuracy_correlation():
    """Plot correlation between average entropy and accuracy"""

    variants = ['Standard', 'No Pos', 'Fewer Heads', 'Small MLP', 'Deep MLP', 'Large Patch']
    accuracies = [7.22, 7.05, 6.90, 7.40, 7.38, 8.54]
    entropies = [5.2513, 5.2513, 5.2513, 5.2512, 5.2512, 5.2507]

    fig, ax = plt.subplots(figsize=(8, 6))

    scatter = ax.scatter(entropies, accuracies, s=200, c=range(len(variants)),
                         cmap='viridis', edgecolor='black', linewidth=2, alpha=0.8)

    for i, variant in enumerate(variants):
        ax.annotate(variant, (entropies[i], accuracies[i]),
                   xytext=(8, 5), textcoords='offset points',
                   fontsize=10, fontweight='bold')

    z = np.polyfit(entropies, accuracies, 1)
    p = np.poly1d(z)
    x_trend = np.linspace(min(entropies)-0.0001, max(entropies)+0.0001, 100)
    ax.plot(x_trend, p(x_trend), '--', color='gray', linewidth=2, alpha=0.7, label=f'Trend: r² = {np.corrcoef(entropies, accuracies)[0,1]**2:.3f}')

    ax.set_xlabel('Average Attention Entropy', fontsize=13, fontweight='bold')
    ax.set_ylabel('Best Validation Accuracy (%)', fontsize=13, fontweight='bold')
    ax.set_title('Entropy-Accuracy Correlation', fontsize=14, fontweight='bold', pad=15)
    ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.8)
    ax.legend(loc='lower left', fontsize=10)
    ax.set_facecolor('#F8F9FA')

    plt.tight_layout()
    plt.savefig('entropy_accuracy_correlation.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

## Training Process

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MaxNLocator

plt.rcParams['font.family'] = 'DejaVu Serif'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.linewidth'] = 1
plt.rcParams['axes.edgecolor'] = 'black'
plt.rcParams['axes.labelcolor'] = 'black'
plt.rcParams['xtick.color'] = 'black'
plt.rcParams['ytick.color'] = 'black'
plt.rcParams['figure.dpi'] = 300


# Standard ViT (from the logs: 7.22% best validation)
standard_train = [1.90, 6.70, 9.10, 14.20, 19.30, 26.70, 36.20, 43.00, 52.80, 61.50,
                  69.40, 74.30, 81.40, 85.30, 87.40, 89.60, 91.00, 92.00, 92.50, 92.60]
standard_val = [2.68, 3.63, 4.41, 4.56, 5.38, 5.68, 5.91, 6.23, 6.05, 5.98,
                6.55, 6.96, 6.81, 6.98, 6.79, 7.22, 7.03, 7.13, 6.93, 6.99]

# No Positional Embeddings (7.05% best validation)
nopos_train = [1.60, 5.90, 10.60, 13.30, 19.70, 27.80, 34.80, 42.10, 51.30, 60.30,
               68.60, 76.90, 81.00, 84.40, 88.20, 89.60, 91.20, 92.40, 92.70, 92.60]
nopos_val = [3.29, 4.56, 4.21, 5.48, 4.85, 5.73, 5.93, 6.20, 6.43, 6.35,
             6.55, 6.23, 6.57, 7.05, 6.65, 6.82, 6.81, 6.63, 6.69, 6.62]

# Fewer Attention Heads (6.90% best validation)
fewerheads_train = [1.20, 5.30, 8.30, 11.90, 18.30, 25.40, 32.30, 42.30, 50.50, 59.60,
                    68.30, 75.10, 79.50, 84.20, 86.70, 88.40, 90.10, 90.90, 91.70, 91.80]
fewerheads_val = [2.62, 3.18, 4.13, 5.01, 5.21, 5.42, 5.86, 5.90, 6.15, 6.42,
                  6.19, 6.44, 6.67, 6.77, 6.75, 6.86, 6.77, 6.84, 6.80, 6.90]

# Small MLP (7.40% best validation)
smallmlp_train = [2.40, 4.80, 8.90, 13.20, 19.40, 26.40, 34.20, 44.40, 52.00, 59.50,
                  65.40, 72.50, 78.80, 83.90, 87.80, 88.40, 90.70, 91.50, 91.90, 92.10]
smallmlp_val = [3.13, 3.79, 5.08, 5.38, 5.86, 6.20, 6.06, 6.70, 6.40, 6.31,
                7.15, 7.04, 7.08, 7.40, 7.11, 7.13, 7.15, 7.22, 7.20, 7.20]

# Deep MLP (7.38% best validation)
deepmlp_train = [1.30, 5.00, 8.10, 14.10, 19.00, 27.00, 34.90, 45.00, 51.70, 62.80,
                 70.50, 76.30, 80.30, 84.30, 87.30, 89.70, 91.30, 91.80, 92.50, 92.50]
deepmlp_val = [2.78, 4.09, 4.67, 4.63, 5.71, 6.10, 6.35, 5.85, 6.62, 6.48,
               7.00, 6.86, 7.38, 6.94, 7.12, 7.05, 7.08, 7.02, 7.16, 7.13]

# Large Patch (8.54% best validation)
largepatch_train = [1.80, 7.50, 11.90, 19.70, 31.90, 41.90, 52.30, 61.10, 72.40, 80.10,
                    87.10, 90.50, 93.10, 96.10, 96.60, 97.60, 98.00, 98.20, 98.30, 98.50]
largepatch_val = [3.63, 4.50, 5.95, 6.43, 6.56, 7.03, 7.54, 7.39, 8.00, 7.91,
                  8.23, 8.44, 8.19, 8.25, 8.26, 8.54, 8.14, 8.26, 8.16, 8.16]

models = [
    {'name': 'Standard ViT', 'train': standard_train, 'val': standard_val, 'best_val': 7.22},
    {'name': 'No Positional Embeddings', 'train': nopos_train, 'val': nopos_val, 'best_val': 7.05},
    {'name': 'Fewer Attention Heads', 'train': fewerheads_train, 'val': fewerheads_val, 'best_val': 6.90},
    {'name': 'Small MLP', 'train': smallmlp_train, 'val': smallmlp_val, 'best_val': 7.40},
    {'name': 'Deep MLP', 'train': deepmlp_train, 'val': deepmlp_val, 'best_val': 7.38},
    {'name': 'Large Patch (32×32)', 'train': largepatch_train, 'val': largepatch_val, 'best_val': 8.54}
]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Training and Validation Curves: ViT Ablation Study (CIFAR-100, 10 shots/class)',
             fontsize=16, fontweight='bold', y=1.02)

train_color = '#1f77b4'
val_color = '#d62728'
best_color = '#2ecc71'

for idx, ax in enumerate(axes.flat):
    model = models[idx]
    epochs = range(1, 21)

    ax.plot(epochs, model['train'], color=train_color, linewidth=2,
            label='Train Accuracy', alpha=0.9)
    ax.plot(epochs, model['val'], color=val_color, linewidth=2,
            label='Validation Accuracy', alpha=0.9)

    best_epoch = model['val'].index(model['best_val']) + 1
    ax.scatter(best_epoch, model['best_val'], color=best_color, s=100,
               edgecolor='black', linewidth=1.5, zorder=5,
               label=f'Best Val: {model["best_val"]:.2f}%')

    ax.axhline(y=model['best_val'], color=best_color, linestyle='--',
               linewidth=1, alpha=0.5)

    ax.set_title(model['name'], fontsize=13, fontweight='bold', pad=10)
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel('Accuracy (%)', fontsize=11)
    ax.set_xlim(1, 20)
    ax.set_ylim(0, 100)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(True, alpha=0.2, linestyle='--', linewidth=0.5)
    ax.tick_params(labelsize=10)


    if idx == 0:
        ax.legend(loc='center right', frameon=True, fancybox=False,
                  edgecolor='black', fontsize=9)

    ax.text(0.02, 0.95, f'Best Val: {model["best_val"]:.2f}%',
            transform=ax.transAxes, fontsize=10, fontweight='bold',
            verticalalignment='top',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                      edgecolor='none', alpha=0.8))


plt.tight_layout()
plt.subplots_adjust(top=0.92, bottom=0.1)
plt.savefig('all_training_curves.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

def plot_summary_table():
    """Create a beautiful summary table of results"""
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.axis('off')
    ax.axis('tight')

    table_data = []
    for model in models:
        table_data.append([
            model['name'],
            f"{model['best_val']:.2f}%",
            f"{model['train'][-1]:.1f}%",
            f"{model['train'][-1] - model['best_val']:.1f}%"
        ])

    table = ax.table(cellText=table_data,
                     colLabels=['Model Variant', 'Best Val Acc (%)', 'Final Train Acc (%)', 'Train-Val Gap (%)'],
                     cellLoc='center',
                     loc='center',
                     colColours=['#4472C4']*4)

    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1.2, 1.5)

    for i, row in enumerate(table_data):
        if '8.54' in row[1]:  # Best
            for j in range(4):
                table[(i+1, j)].set_facecolor('#d4edda')  # light green
        elif '6.90' in row[1]:  # Worst
            for j in range(4):
                table[(i+1, j)].set_facecolor('#f8d7da')  # light red

    plt.title('ViT Ablation Study: Summary of Results (10 shots/class on CIFAR-100)',
              fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig('summary_table.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()


## Attention Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import timm
import os
import pandas as pd

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.linewidth'] = 1
plt.rcParams['axes.edgecolor'] = 'black'
plt.rcParams['axes.labelcolor'] = 'black'
plt.rcParams['xtick.color'] = 'black'
plt.rcParams['ytick.color'] = 'black'
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300


class AttentionAnalyzer:
    def __init__(self, model):
        self.model = model
        self.attention_maps = []
        self.hooks = []
        self._register_hooks()

    def _hook_fn(self, module, input, output):
        self.attention_maps.append(output.detach().cpu())

    def _register_hooks(self):
        if hasattr(self.model, 'blocks'):
            for block in self.model.blocks:
                if hasattr(block, 'attn'):
                    hook = block.attn.register_forward_hook(self._hook_fn)
                    self.hooks.append(hook)

    def clear(self):
        self.attention_maps = []

    def remove_hooks(self):
        for hook in self.hooks:
            hook.remove()

    def get_attention_maps(self, images):
        self.model.eval()
        self.clear()
        with torch.no_grad():
            _ = self.model(images)
        maps = self.attention_maps.copy()
        self.clear()
        return maps


def load_model(variant='standard', n_per_class=10, device='cpu'):
    """Load pretrained model for a specific variant"""
    model = timm.create_model(
        'vit_tiny_patch16_224',
        pretrained=False,
        num_classes=100,
        img_size=32
    )

    if variant == 'no_pos':
        if hasattr(model, 'pos_embed'):
            model.pos_embed.data.zero_()
    elif variant == 'fewer_heads':
        model = timm.create_model(
            'vit_tiny_patch16_224',
            pretrained=False,
            num_classes=100,
            img_size=32,
            num_heads=2
        )
    elif variant == 'small_mlp':
        for block in model.blocks:
            if hasattr(block, 'mlp') and hasattr(block.mlp, 'drop'):
                block.mlp.drop.p = 0.3
    elif variant == 'large_patch':
        model = timm.create_model(
            'vit_tiny_patch16_224',
            pretrained=False,
            num_classes=100,
            img_size=32,
            patch_size=32
        )

    model_path = f'vit_{variant}_n{n_per_class}.pth'
    if os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path, map_location=device))
        print(f"Loaded {variant} model from {model_path}")
    else:
        print(f"No pretrained model found for {variant}")

    return model.to(device)


def get_test_images(n_per_class=10, num_images=1):
    """Load a single test image from CIFAR-100"""
    np.random.seed(42)
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ])

    test_set = torchvision.datasets.CIFAR100(
        root='./data', train=False, download=True, transform=transform
    )

    idx = np.random.randint(0, len(test_set))
    img, label = test_set[idx]

    return img.unsqueeze(0), label, test_set.classes[label]



def print_attention_matrices(model, image, device, variant_name='standard', save_path=None):

    analyzer = AttentionAnalyzer(model)
    attention_maps = analyzer.get_attention_maps(image.to(device))

    if not attention_maps:
        print("No attention maps captured")
        return

    num_layers = len(attention_maps)
    num_heads = attention_maps[0].shape[1]

    print("\n" + "="*80)
    print(f"ATTENTION MATRICES - {variant_name.upper()}")
    print("="*80)
    print(f"Input image class: {variant_name}_class")

    if save_path:
        f = open(save_path, 'w')
        f.write(f"ATTENTION MATRICES - {variant_name.upper()}\n")
        f.write("="*80 + "\n")

    for layer_idx in range(num_layers):
        attn = attention_maps[layer_idx]

        print(f"\n{'─'*80}")
        print(f"LAYER {layer_idx + 1}")
        print(f"{'─'*80}")

        if save_path:
            f.write(f"\n{'─'*80}\n")
            f.write(f"LAYER {layer_idx + 1}\n")
            f.write(f"{'─'*80}\n")

        if len(attn.shape) == 4:
            seq_len = attn.shape[2]

            for head_idx in range(num_heads):
                attn_matrix = attn[0, head_idx, :, :].cpu().numpy()

                print(f"\n  Head {head_idx + 1} (CLS attention to patches):")
                print(f"  Matrix shape: {attn_matrix.shape}")
                print(f"  Min: {attn_matrix.min():.6f}, Max: {attn_matrix.max():.6f}, Mean: {attn_matrix.mean():.6f}")
                print(f"  CLS token attention distribution:")

                cls_attn = attn_matrix[0, 1:]  # Skip CLS self-attention

                grid_size = int(np.sqrt(cls_attn.shape[0]))
                if grid_size * grid_size == cls_attn.shape[0]:
                    attn_grid = cls_attn.reshape(grid_size, grid_size)

                    print(f"\n  Attention weights from CLS to {grid_size}×{grid_size} patches:")
                    print("  " + "─" * (grid_size * 10))
                    for i in range(grid_size):
                        row_str = "  "
                        for j in range(grid_size):
                            row_str += f"{attn_grid[i, j]:8.4f} "
                        print(row_str)

                    probs = np.clip(cls_attn, 1e-10, 1.0)
                    entropy = -np.sum(probs * np.log(probs))
                    print(f"\n  Entropy: {entropy:.6f}")

                    if save_path:
                        f.write(f"\n  Head {head_idx + 1}\n")
                        f.write(f"  Min: {attn_matrix.min():.6f}, Max: {attn_matrix.max():.6f}, Mean: {attn_matrix.mean():.6f}\n")
                        f.write(f"  Attention weights ({grid_size}×{grid_size}):\n")
                        for i in range(grid_size):
                            row_str = "  "
                            for j in range(grid_size):
                                row_str += f"{attn_grid[i, j]:8.4f} "
                            f.write(row_str + "\n")
                        f.write(f"  Entropy: {entropy:.6f}\n")

        elif len(attn.shape) == 3:
            seq_len = attn.shape[2]

            for head_idx in range(num_heads):
                attn_weights = attn[0, head_idx, 1:].cpu().numpy()

                print(f"\n  Head {head_idx + 1}:")
                print(f"  Shape: {attn_weights.shape}")
                print(f"  Min: {attn_weights.min():.6f}, Max: {attn_weights.max():.6f}, Mean: {attn_weights.mean():.6f}")

                grid_size = int(np.sqrt(attn_weights.shape[0]))
                if grid_size * grid_size == attn_weights.shape[0]:
                    attn_grid = attn_weights.reshape(grid_size, grid_size)

                    print(f"\n  Attention weights ({grid_size}×{grid_size}):")
                    print("  " + "─" * (grid_size * 10))
                    for i in range(grid_size):
                        row_str = "  "
                        for j in range(grid_size):
                            row_str += f"{attn_grid[i, j]:8.4f} "
                        print(row_str)

                    probs = np.clip(attn_weights, 1e-10, 1.0)
                    entropy = -np.sum(probs * np.log(probs))
                    print(f"\n  Entropy: {entropy:.6f}")

                    if save_path:
                        f.write(f"\n  Head {head_idx + 1}\n")
                        f.write(f"  Min: {attn_weights.min():.6f}, Max: {attn_weights.max():.6f}, Mean: {attn_weights.mean():.6f}\n")
                        f.write(f"  Attention weights ({grid_size}×{grid_size}):\n")
                        for i in range(grid_size):
                            row_str = "  "
                            for j in range(grid_size):
                                row_str += f"{attn_grid[i, j]:8.4f} "
                            f.write(row_str + "\n")
                        f.write(f"  Entropy: {entropy:.6f}\n")

    print("\n" + "="*80)

    if save_path:
        f.close()
        print(f"\nAttention matrices saved to: {save_path}")

    analyzer.remove_hooks()
    return attention_maps

def print_all_variant_matrices(n_per_class=10, device='cpu'):
    """Print attention matrices for all 6 variants"""

    variants = [
        {'name': 'Standard ViT', 'key': 'standard'},
        {'name': 'No Positional Embeddings', 'key': 'no_pos'},
        {'name': 'Fewer Attention Heads', 'key': 'fewer_heads'},
        {'name': 'Small MLP', 'key': 'small_mlp'},
        {'name': 'Deep MLP', 'key': 'deep_mlp'},
        {'name': 'Large Patch (32×32)', 'key': 'large_patch'}
    ]

    print("\nLoading test image from CIFAR-100...")
    image, label, class_name = get_test_images(n_per_class, num_images=1)
    print(f"   Using image of class: {class_name} (label: {label})")

    for variant in variants:
        print(f"\n{'='*80}")
        print(f"PROCESSING: {variant['name']}")
        print(f"{'='*80}")

        model = load_model(variant['key'], n_per_class, device)


        save_path = f'attention_matrices_{variant["key"]}.txt'
        attn_maps = print_attention_matrices(
            model, image, device,
            variant_name=variant['key']
        )

    print("\n" + "="*80)
    print("All attention matrices generated successfully!")
    print("="*80)
    # print("\nGenerated files:")
    # for variant in variants:
    #     print(f"  • attention_matrices_{variant['key']}.txt")

if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print("="*80)
    print("VIT ATTENTION MATRICES - NUMERICAL VALUES")
    print("="*80)
    print(f"Device: {device}")
    print(f"Displaying attention weights as numerical matrices for each layer and head")

    # Run for all 6 variants
    print_all_variant_matrices(n_per_class=10, device=device)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.family'] = 'DejaVu Serif'
plt.rcParams['font.size'] = 11
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300

# The mean of each head at every layer, extracted from the actual logs."

# 1. Standard ViT
standard_matrix = np.array([
    [-0.003046, -0.001263, -0.001055, -0.000960, -0.000838],  # Layer 1
    [0.000351, 0.000047, 0.000286, 0.000841, 0.000492],       # Layer 2
    [-0.002426, -0.001529, -0.001174, -0.001864, -0.002236],   # Layer 3
    [0.002735, 0.002780, 0.002881, 0.002564, 0.002317],        # Layer 4
    [-0.001405, -0.002359, -0.002465, -0.001814, -0.001992],   # Layer 5
    [0.002905, 0.001945, 0.002192, 0.003157, 0.002067],        # Layer 6
    [-0.005045, -0.005358, -0.005437, -0.005103, -0.004945],   # Layer 7
    [0.000530, 0.000586, 0.000813, 0.000494, 0.000315],        # Layer 8
    [0.005700, 0.006850, 0.006743, 0.006035, 0.006642],        # Layer 9
    [0.000616, 0.001106, 0.000995, 0.000785, 0.000801],        # Layer 10
    [-0.000045, -0.000225, 0.000058, 0.000161, -0.000097],     # Layer 11
    [0.002883, 0.002929, 0.003385, 0.002335, 0.002743]         # Layer 12
])

# 2. No Positional Embeddings
nopos_matrix = np.array([
    [0.000286, -0.000915, -0.000695, -0.000743, -0.000989],   # Layer 1
    [0.003977, 0.004068, 0.004176, 0.004125, 0.003709],        # Layer 2
    [0.000338, 0.000181, 0.000260, 0.000469, 0.000352],        # Layer 3
    [0.002845, 0.003145, 0.002952, 0.002336, 0.003021],        # Layer 4
    [-0.001588, -0.000271, -0.000692, -0.002015, -0.000581],   # Layer 5
    [-0.002716, -0.003241, -0.003177, -0.003259, -0.003152],   # Layer 6
    [0.000536, 0.000514, 0.000593, 0.000398, 0.000851],        # Layer 7
    [-0.005970, -0.005979, -0.005869, -0.005886, -0.006132],   # Layer 8
    [0.002805, 0.001973, 0.002220, 0.003341, 0.002270],        # Layer 9
    [-0.003138, -0.002787, -0.002938, -0.003019, -0.002881],   # Layer 10
    [0.000237, 0.000321, 0.000089, 0.000009, 0.000380],        # Layer 11
    [0.001695, 0.002053, 0.002070, 0.001494, 0.001782]         # Layer 12
])

# 3. Fewer Attention Heads
fewerheads_matrix = np.array([
    [-0.000409, 0.000411, 0.000424, 0.000204, 0.000181],       # Layer 1
    [0.004243, 0.003700, 0.004045, 0.003911, 0.003907],        # Layer 2
    [0.002192, 0.002676, 0.002684, 0.002474, 0.002494],        # Layer 3
    [-0.000587, -0.000646, -0.000648, -0.000551, -0.000787],   # Layer 4
    [0.002071, 0.001920, 0.002023, 0.002179, 0.002376],        # Layer 5
    [0.003249, 0.003325, 0.003318, 0.002573, 0.002919],        # Layer 6
    [-0.003041, -0.003147, -0.002871, -0.002369, -0.004250],   # Layer 7
    [0.003109, 0.002774, 0.002830, 0.002944, 0.002965],        # Layer 8
    [-0.000677, -0.000503, -0.000369, -0.000233, -0.000545],   # Layer 9
    [0.001572, 0.000528, 0.000135, -0.000301, -0.000373],      # Layer 10
    [-0.001912, -0.001491, -0.001597, -0.001982, -0.001743],   # Layer 11
    [-0.003079, -0.003316, -0.003332, -0.003094, -0.003116]    # Layer 12
])

# 4. Small MLP
smallmlp_matrix = np.array([
    [-0.003358, -0.002543, -0.002474, -0.001951, -0.002253],   # Layer 1
    [-0.001546, -0.001608, -0.001583, -0.001279, -0.001411],   # Layer 2
    [0.001959, 0.002376, 0.002532, 0.002511, 0.002013],        # Layer 3
    [0.002802, 0.002462, 0.002334, 0.001701, 0.002430],        # Layer 4
    [0.002072, 0.001330, 0.001056, 0.002017, 0.001552],        # Layer 5
    [0.000409, 0.000273, 0.000398, 0.000356, 0.000482],        # Layer 6
    [-0.003118, -0.002373, -0.002403, -0.002668, -0.002555],   # Layer 7
    [-0.000262, -0.000061, 0.000059, -0.000227, -0.000378],    # Layer 8
    [0.007056, 0.007348, 0.007423, 0.007007, 0.006947],        # Layer 9
    [-0.008022, -0.008023, -0.008044, -0.008040, -0.008097],   # Layer 10
    [0.004418, 0.003057, 0.003270, 0.004548, 0.004331],        # Layer 11
    [0.008754, 0.009261, 0.008902, 0.008096, 0.008689]         # Layer 12
])

# 5. Deep MLP
deepmlp_matrix = np.array([
    [-0.001587, -0.003355, -0.003547, -0.002562, -0.002578],   # Layer 1
    [-0.005146, -0.005021, -0.005163, -0.005078, -0.005004],   # Layer 2
    [0.000098, 0.000997, 0.001015, 0.000631, 0.000558],        # Layer 3
    [-0.001182, -0.001000, -0.001070, -0.001073, -0.001114],   # Layer 4
    [0.001855, 0.002560, 0.002652, 0.002534, 0.002595],        # Layer 5
    [-0.002689, -0.002395, -0.002469, -0.002434, -0.002371],   # Layer 6
    [0.003423, 0.003208, 0.003298, 0.003080, 0.003294],        # Layer 7
    [-0.001498, -0.001619, -0.001277, -0.001600, -0.001758],   # Layer 8
    [-0.003268, -0.003150, -0.002950, -0.002717, -0.002818],   # Layer 9
    [-0.001970, -0.001075, -0.001228, -0.001733, -0.001663],   # Layer 10
    [0.001588, 0.001268, 0.001380, 0.001726, 0.001572],        # Layer 11
    [-0.002133, -0.001661, -0.001771, -0.002073, -0.001680]    # Layer 12
])

# 6. Large Patch (2 heads)
largepatch_raw = np.array([
    [-0.004553, -0.005384],   # Layer 1
    [-0.002852, -0.004995],   # Layer 2
    [-0.003436, -0.003201],   # Layer 3
    [-0.002579, -0.002915],   # Layer 4
    [0.001385, 0.000047],     # Layer 5
    [-0.006292, -0.005762],   # Layer 6
    [-0.005635, -0.002995],   # Layer 7
    [0.004043, 0.003123],     # Layer 8
    [0.007367, 0.007287],     # Layer 9
    [0.001730, -0.000526],    # Layer 10
    [0.005644, 0.001958],     # Layer 11
    [0.004493, 0.004627]      # Layer 12
])
# to 5 heads
largepatch_matrix = np.full((12, 5), np.nan)
largepatch_matrix[:, :2] = largepatch_raw

matrices = [
    (standard_matrix, 'Standard ViT', 7.22),
    (nopos_matrix, 'No Positional Embeddings', 7.05),
    (fewerheads_matrix, 'Fewer Attention Heads', 6.90),
    (smallmlp_matrix, 'Small MLP', 7.40),
    (deepmlp_matrix, 'Deep MLP', 7.38),
    (largepatch_matrix, 'Large Patch (32×32)', 8.54)
]


# heatmap
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Attention Score Distribution Across Layers and Heads: All ViT Variants\n(10 shots/class on CIFAR-100)',
             fontsize=16, fontweight='bold', y=1)

all_values = np.concatenate([m[0].flatten() for m in matrices if m[0] is not None])
all_values = all_values[~np.isnan(all_values)]
vmin, vmax = -0.01, 0.01

for idx, ax in enumerate(axes.flat):
    matrix, title, acc = matrices[idx]
    masked_matrix = np.ma.masked_where(np.isnan(matrix), matrix)

    im = ax.imshow(masked_matrix, cmap='RdBu_r', aspect='auto',
                   vmin=vmin, vmax=vmax, interpolation='nearest')

    ax.set_title(f'{title}\nBest Val Acc: {acc}%', fontsize=12, fontweight='bold', pad=10)

    if idx >= 3:
        ax.set_xlabel('Attention Head', fontsize=11)
    if idx % 3 == 0:
        ax.set_ylabel('Layer', fontsize=11)

    ax.set_xticks(range(5))
    ax.set_xticklabels([f'H{i+1}' for i in range(5)], fontsize=9)
    ax.set_yticks(range(12))
    ax.set_yticklabels([f'L{i+1}' for i in range(12)], fontsize=9)

    for i in range(12):
        for j in range(5):
            if not np.isnan(matrix[i, j]):
                value = matrix[i, j]
                text_color = 'white' if abs(value) > 0.006 else 'black'
                ax.text(j, i, f'{value:.4f}', ha='center', va='center',
                        fontsize=7, color=text_color)

    if 'Large Patch' in title:
        ax.text(3.5, 6, 'Only 2 heads\n(others not present)',
                ha='center', va='center', fontsize=9,
                bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))

    for i in range(1, 12):
        ax.axhline(y=i-0.5, color='gray', linestyle='-', linewidth=0.3, alpha=0.3)
    for j in range(1, 5):
        ax.axvline(x=j-0.5, color='gray', linestyle='-', linewidth=0.3, alpha=0.3)

cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label('Mean Attention Score (pre-softmax)', rotation=270, labelpad=25, fontsize=12)

plt.tight_layout()
plt.subplots_adjust(right=0.9, top=0.92, bottom=0.05)
plt.savefig('figure_all_variants_attention_heatmaps.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("\n" + "="*80)
print("ATTENTION MATRIX STATISTICS - ALL VARIANTS")
print("="*80)

for matrix, title, acc in matrices:
    valid_values = matrix[~np.isnan(matrix)]
    print(f"\n{title} (Acc: {acc}%):")
    print(f"  Mean: {valid_values.mean():.6f}")
    print(f"  Std: {valid_values.std():.6f}")
    print(f"  Min: {valid_values.min():.6f}")
    print(f"  Max: {valid_values.max():.6f}")